# Ollama Local Model Usage
This notebook uses Ollama on your local machine. No Gemini or OpenAI cloud API key is required for local Ollama models.

## Requirements
- Install Ollama locally: https://ollama.com/
- Pull a local model, for example: `ollama pull phi3`
- Confirm the `ollama` CLI works in your environment

In [53]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

### Generating Text

In [54]:
def generate_text(prompt):
    response = client.chat.completions.create(
        model="phi3",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=10
    )
    return response.choices[0].message.content

In [55]:
prompt = "Once upon a time"

In [56]:
generated_text = generate_text(prompt)
print(prompt, generated_text)

Once upon a time "In the heart of an ancient forest, where


### Customizing the Output

In [57]:
def generate_text(prompt, max_tokens, temperature):
    response = client.chat.completions.create(
        model="phi3",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

In [58]:
generated_text = generate_text(prompt, 50, 0)
print(prompt, generated_text)

Once upon a time "Once upon a time, in the heart of an ancient forest shrouded by mist and mystery, there lived a young girl named Elara. Her days were spent wandering among towering trees whose roots whispered secrets older than her grand


In [59]:
generated_text = generate_text(prompt, 50, 1)
print(prompt, generated_text)

Once upon a time Certainty prevails that "eddy currents are generated in conductors when they move through magnetic fields, inducing undesirable effects such as heating and losses" encapsulates the essence of your question on an engineering level


### Summarizing Text

In [60]:
def text_summarizer(prompt):
    response = client.chat.completions.create(
        model="phi3",
        messages=[
            {
                "role": "system",
                "content": "Extract a list of keywords from the text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5,
        max_tokens=256
    )

    return response.choices[0].message.content.strip()

In [61]:
prompt = "Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more."
print(prompt)

Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more.


In [62]:
result = text_summarizer(prompt)
print(result)

1. Master Reef Guide Kirsty Whitman
2. Snorkel mask
3. Female manta ray
4. Male manta ray
5. Potential mate
6. Underwater ballet
7. Snorkelling safari
8. Current (undersea) 
9. Marine life encounter
1 endocrine disruptor and its impact on the reproductive system of marine animals, specifically fishes that are part of a coral-reef ecosystem like manta rays:

Underwater ballet - The term "underwater ballet" is used metaphorically to describe the graceful movements exhibited by male and female manta rays in their courtship behavior. This phrase does not directly relate to endocrine disruptors or reproductive systems but serves as a poetic way of framing the natural interactions observed during snorkeling activities related to marine life, particularly within coral-reef ecosystems where such behaviors are commonplace among manta ray species.

Snorkel mask - A piece of equipment used for observing underwater environments and is not directly linked to endocrine disruptors or reproduct


### Poetic Chatbot

In [63]:
def poetic_chatbot(prompt):
    response = client.chat.completions.create(
        model="phi3",
        messages=[
            {
                "role": "system",
                "content": "You are a poetic chatbot. Always answer in short poetic form."
            },
            {
                "role": "user",
                "content": "When was Google founded?"
            },
            {
                "role": "assistant",
                "content": "In the late '90s, a spark did ignite, Google emerged, a radiant light..."
            },
            {
                "role": "user",
                "content": "Which country has the youngest president?"
            },
            {
                "role": "assistant",
                "content": "Ah, the pursuit of youth in politics..."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=1,
        max_tokens=256
    )

    return response.choices[0].message.content.strip()

In [64]:
prompt = "When was cheese first made?"
poetic_chatbot(prompt)

"A time long past and hard to trace, but tales tell its ancient embrace. Cheeses formed by goats' milk graced tables then—crafted with fervor since days gone a-wilting. In the earliest records found in Asia Minor (ancient Turkey), around six thousand years ago..."

In [65]:
prompt = "What is the next course to be uploaded to 365DataScience?"
poetic_chatbot(prompt)

"Next sails forth a dish of predictive modeling's bliss; machine learning flavors infused within. Look for patterns like secret sauce, where algorithms do dresses—a feast on data tables. Ready yourself with notebook in hand, your journey through analytics stands proudly grand!"

### Langchain

In [66]:
# import sys
# !{sys.executable} -m pip install langchain==0.0.352 langchain-community langchain-text-splitters faiss-cpu beautifulsoup4 langchain-core


from langchain_community.llms import Ollama
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage


In [67]:
llm = Ollama(model="phi3")

In [68]:
loader = WebBaseLoader("https://365datascience.com/upcoming-courses")
docs = loader.load()

### Load & split documents

In [69]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

### Embeddings + Vector DB

In [70]:
embeddings = OllamaEmbeddings(model="phi3")

db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever()

### Format docs function

In [71]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Prompt template (with memory)

In [72]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based only on the given context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

### LCEL RAG Chain

In [73]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "chat_history": lambda x: x.get("chat_history", [])
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Memory store

In [74]:
chat_history = []

In [75]:
def ask(question):
    global chat_history

    response = chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    # update memory
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=response)
    ])

    return response

In [ ]:
print(ask("What is this document about?"))
print(ask("Can you explain more?"))


The document appears to be a comprehensive resource compilation aimed at individuals seeking knowledge and skills in various aspects related to technology, specifically for those interested in data science, machine learning, Python programming, SQL/Tableau analysis, coding interviews, thinking like a data scientist, introductory shell commands, customer behavior studies using business intelligence tools, as well as technical interview preparation. The resources within the document are highly rated and cover diverse topics such as convolutional neural networks (CNNs) in TensorFlow, speech recognition with Python, SQL/Tableau-based analyses of company churn or engagement rates among customers, coding templates for efficient programming practices alongside infographics summarizing key concepts to aid understanding. Additionally, there's a particular focus on interview preparation resources like 'Python Coding Interview Prep,' which specifically targets individuals looking to excel in tech